# Imports

In [1]:
import importlib
import sys
import torch

sys.path.insert(0, '../..')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../../load/event_log_loader')

import new_event_log_loader

# Data

### Load Data Files

In [2]:
# Path to your pickle file (saved with torch.save)
file_path_train = '../../../../../load/encoded_data/PCR_1_train.pkl'
# Load the dataset using torch.load
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_train_dataset))

# Path to your pickle file (saved with torch.save)
file_path_val = '../../../../../load/encoded_data/PCR_1_val.pkl'
# Load the dataset using torch.load
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_val_dataset))


<class 'new_event_log_loader.EventLogDataset'>
<class 'new_event_log_loader.EventLogDataset'>


### Train Data Insights

In [3]:
# Helpdesk Dataset Categories, Features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
print(helpdesk_all_categories_cat)

helpdesk_all_categories_num = helpdesk_all_categories[1]
print(helpdesk_all_categories_num)

for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk (5) Categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Total Amount of Category labels: {cat[1]}")

print('\n')    

for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk (5) Numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Amount Numerical: {num[1]}")
     
# Get concept_name id:
# 
concept_name = 'concept:name'
concept_name_id = [i for i, cat in enumerate(helpdesk_all_categories[0]) if cat[0] == concept_name][0]

print("ID concet name in cat list: ", concept_name_id)

duration_seconds = 'duration_seconds'
duration_seconds_id = [i for i, num in enumerate(helpdesk_all_categories[1]) if num[0] == duration_seconds][0]
print("ID duration_seconds in num list: ", duration_seconds_id)

[('concept:name', 9, {'Callback timeout': 1, 'Export result': 2, 'Export to EMS': 3, 'Match patient data': 4, 'Receive sample state': 5, 'Send notification': 6, 'Wait for plate validation': 7, 'timeout': 8})]
[('seconds_in_day', 1, {}), ('day_in_week', 1, {}), ('duration_seconds', 1, {})]
Helpdesk (5) Categorical feature: concept:name, Index position in categorical data list: 0
Helpdesk (5) Total Amount of Category labels: 9


Helpdesk (5) Numerical feature: seconds_in_day, Index position in categorical data list: 0
Helpdesk (5) Amount Numerical: 1
Helpdesk (5) Numerical feature: day_in_week, Index position in categorical data list: 1
Helpdesk (5) Amount Numerical: 1
Helpdesk (5) Numerical feature: duration_seconds, Index position in categorical data list: 2
Helpdesk (5) Amount Numerical: 1
ID concet name in cat list:  0
ID duration_seconds in num list:  2


In [4]:
selected_cat_attributes = ['concept:name']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

selected_categories = (
    [cat for cat in helpdesk_all_categories[0] if cat[0] in selected_cat_attributes],
    [num for num in helpdesk_all_categories[1] if num[0] in selected_num_attributes]
)

# Training Configuration

In [5]:
import stochasticLSTM.model

importlib.reload(stochasticLSTM.model)
from stochasticLSTM.model import StochasticLSTM

"""
Specific model parameters from paper: 
"""

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")

# Size hidden layer
hidden_size = 128

# Number of LSTM cells
num_layers = 2

# Fixed Dropout probability
p_fix = 0.1

# Lambda for L2 (weight, bias, dropout) regularization: According to formula: 1/2N
regularization_term = 1e-5

# Hans Weytjens LSTM model
model = StochasticLSTM(
    data_set_categories=helpdesk_all_categories,
    model_input_feat=selected_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    weight_reg=regularization_term,
    p_fix=p_fix,
    device=device,
)

import loss.losses

importlib.reload(loss.losses)
from loss.losses import Loss

loss_obj = Loss()


import training.train

importlib.reload(training.train)
from training.train import Training

from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(comment="train")


"""
Parameter of Probabilistic Suffix Prediction experimental design, to ensure fair comparison:
"""

# Start learning rate
learning_rate = 5e-3

# Optimizer and Scheduler
optimizer = torch.optim.Adam(
    params=model.parameters(), lr=learning_rate, weight_decay=0
)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=20, min_lr=1e-10
)

# Epochs
num_epochs = 200

# Batch of model input
batch_size = 128

# shuffle data
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

trainer = Training(
    model=model,
    device=device,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    selected_features=(selected_cat_attributes, selected_num_attributes),
    concept_name_id=concept_name_id,
    duration_seconds_id=duration_seconds_id,
    loss_obj=loss_obj,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path="model.pkl",
)

# Train the model:
trainer.train()

Embeddings:  ModuleList(
  (0): Embedding(9, 16)
)
Total embedding feature size:  16
Input feature size:  18
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1


Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7fc638a86d70>
Epochs:  200
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/174 [00:00<?, ?it/s]

Epoch [1/200], Learning Rate: 0.005
Training: Avg Attenuated Training Loss: 0.7535
Validation: Avg Standard Validation Loss: 0.9572
Validation: Avg Attenuated Validation Loss: 5.3380
Validation Loss for Scheduler: 0.9572
saving model


  0%|          | 0/174 [00:00<?, ?it/s]

...

Epoch [198/200], Learning Rate: 0.00015625
Training: Avg Attenuated Training Loss: -4.3571
Validation: Avg Standard Validation Loss: 0.8699
Validation: Avg Attenuated Validation Loss: -4.9331
Validation Loss for Scheduler: 0.8699
saving model


  0%|          | 0/174 [00:00<?, ?it/s]

Epoch [199/200], Learning Rate: 0.00015625
Training: Avg Attenuated Training Loss: -4.3455
Validation: Avg Standard Validation Loss: 0.8824
Validation: Avg Attenuated Validation Loss: -4.2081
Validation Loss for Scheduler: 0.8824
saving model


  0%|          | 0/174 [00:00<?, ?it/s]

Epoch [200/200], Learning Rate: 0.00015625
Training: Avg Attenuated Training Loss: -4.3669
Validation: Avg Standard Validation Loss: 0.9730
Validation: Avg Attenuated Validation Loss: -4.5688
Validation Loss for Scheduler: 0.9730
saving model
Training complete.
Model saved to path: model.pkl
